# INTELIGÊNCIA ARTIFICIAL
## LABORATÓRIO # 03 — EDA (Exploratory Data Analysis)
**Do primeiro olhar à decisão · Análise completa do dataset CarPrice**

---

### Objetivos do Laboratório
1. **Compreensão dos Dados (Data Understanding):** Estrutura, tipos, primeiras linhas e distribuições.
2. **Preparação e Limpeza:** Tratamento de valores faltantes, identificação de duplicatas e colunas espúrias.
3. **Visualizações Estatísticas:** Distribuições univariadas, relações bivariadas com a variável alvo (`price`).
4. **Resolução de Desafios:**
   - *Desafio 1:* Extração e limpeza do nome da marca a partir de `CarName`.
   - *Desafio 2:* Identificação e correção de grafias erradas (*typos*) na marca.
   - *Desafio 3:* Gráfico de tendência/dispersão entre variáveis de alta correlação.
5. **Feature Engineering:** Criação de variáveis derivadas úteis para modelos preditivos.


### Importação das Bibliotecas e Leitura dos Dados

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo dos gráficos
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams['figure.figsize'] = (10, 5)

# Leitura do dataset CarPrice_Assignment.csv com caminho relativo
caminho = "CarPrice_Assignment.csv"
df = pd.read_csv(caminho, sep=",")

print(f"Dataset carregado com sucesso! Dimensões: {df.shape}")
df.head()

---
### Fase 1 — Compreensão dos Dados (Data Understanding)
Examinamos a estrutura, os tipos de variáveis (numéricas vs. categóricas) e as estatísticas gerais.

In [ ]:
print("=== INFORMAÇÕES GERAIS E TIPOS ===")
df.info()

In [ ]:
print("=== ESTATÍSTICA DAS VARIÁVEIS NUMÉRICAS ===")
display(df.describe())

In [ ]:
print("=== ESTATÍSTICA DAS VARIÁVEIS CATEGÓRICAS (OBJECT) ===")
display(df.describe(include="O"))

In [ ]:
# Inspeção de valores únicos e frequências de colunas categóricas chave
print("Tipos de carroceria (carbody):", list(df['carbody'].unique()))
print("\nContagem por tipo de carroceria:")
print(df['carbody'].value_counts())

print("\nContagem por sistema de combustível (fuelsystem):")
print(df['fuelsystem'].value_counts())

---
### Fase 2 — Preparação dos Dados (Limpeza e Decisões)
Detectamos valores ausentes e duplicatas e tomamos decisões fundamentadas.

In [ ]:
# 1. Verificando valores faltantes por coluna
valores_nulos = df.isnull().sum()
print("Total de valores nulos por coluna:")
print(valores_nulos[valores_nulos > 0] if valores_nulos.sum() > 0 else "Nenhum valor nulo encontrado no dataset original!")

# Exemplo didático do roteiro: tratamento com mediana (robusta a outliers)
mediana_preco = df['price'].median()
df['price'] = df['price'].fillna(mediana_preco)
print(f"Mediana de preço calculada: R$ {mediana_preco:.2f}")

In [ ]:
# 2. Identificação de linhas duplicadas
duplicatas = df[df.duplicated(keep='first')]
print(f"Linhas duplicadas identificadas: {len(duplicatas)}")
if len(duplicatas) > 0:
    df.drop_duplicates(keep='first', inplace=True)
    print("Duplicatas removidas com sucesso.")

---
### Fase 3 — Visualização e Relações

#### 3.1 Distribuição da Variável Alvo (`price`)
Avaliamos a assimetria e o formato da distribuição do preço dos carros.

In [ ]:
plt.figure(figsize=(10, 4))
fig = sns.histplot(data=df, x='price', kde=True, color='#2b5c8f')
fig.set_title('Distribuição da Variável Alvo (Price)', fontsize=15, pad=10)
fig.set_xlabel('Preço ($)', fontsize=12)
fig.set_ylabel('Frequência', fontsize=12)
plt.axvline(df['price'].mean(), color='red', linestyle='--', label=f'Média: ${df["price"].mean():.0f}')
plt.axvline(df['price'].median(), color='green', linestyle='-', label=f'Mediana: ${df["price"].median():.0f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Média: {df['price'].mean():.2f} | Mediana: {df['price'].median():.2f}")
print("Observação: A cauda longa à direita (assimetria positiva) puxa a média para cima da mediana devido a carros de luxo.")

#### 3.2 Distribuição de Variáveis Categóricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.countplot(data=df, x='carbody', ax=axes[0], palette='Blues_r')
axes[0].set_title('Contagem por Tipo de Carroceria (carbody)', fontsize=13)
axes[0].set_xlabel('Carroceria')
axes[0].set_ylabel('Quantidade')

sns.countplot(data=df, x='enginelocation', ax=axes[1], palette='Oranges_r')
axes[1].set_title('Localização do Motor (enginelocation)', fontsize=13)
axes[1].set_xlabel('Posição do Motor')
axes[1].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

#### 3.3 Desafio 1 — Gráfico Legível das Marcas de Veículos
O atributo `CarName` possui muitos nomes compostos com modelo e versão, tornando o gráfico ilegível.
**Solução:** Extrair a marca (primeira palavra) com `.str.split(' ').str[0]` e plotar no eixo vertical (`y='brand'`).

In [ ]:
# Extraindo a marca
df['brand'] = df['CarName'].str.split(' ').str[0].str.lower()

plt.figure(figsize=(10, 8))
ordem_marcas = df['brand'].value_counts().index
sns.countplot(data=df, y='brand', order=ordem_marcas, palette='viridis')
plt.title('Desafio 1: Contagem por Marca do Carro (Legível no Eixo Y)', fontsize=14)
plt.xlabel('Quantidade')
plt.ylabel('Marca')
plt.tight_layout()
plt.show()

#### 3.4 Desafio 2 — Correção de Grafias Incorretas (*Typos*) nas Marcas
Analisando os valores únicos de `brand`, encontramos nomes com erros como `maxda`, `toyouta`, `vokswagen`, `vw`, `porcshce`.

In [ ]:
print("Marcas únicas ANTES da correção:")
print(sorted(df['brand'].unique()))

# Mapeamento de correções solicitado no laboratório
mapa_correcoes = {
    'maxda': 'mazda',
    'toyouta': 'toyota',
    'vokswagen': 'volkswagen',
    'vw': 'volkswagen',
    'porcshce': 'porsche'
}
df['brand'] = df['brand'].replace(mapa_correcoes)

print("\nMarcas únicas APÓS a correção:")
print(sorted(df['brand'].unique()))

#### 3.5 Remoção de Coluna de ID e Análise de Outliers
A coluna `car_ID` é meramente sequencial e não traz informação preditiva.

In [ ]:
if 'car_ID' in df.columns:
    df.drop('car_ID', axis=1, inplace=True)
    print("Coluna 'car_ID' removida com sucesso!")

# Boxplot de Preço para identificação de outliers
plt.figure(figsize=(8, 3))
sns.boxplot(data=df, x='price', color='#5dade2')
plt.title('Boxplot de Preço — Identificação de Outliers', fontsize=13)
plt.xlabel('Preço ($)')
plt.tight_layout()
plt.show()

# Cálculo do intervalo interquartil (IQR)
q1 = df['price'].quantile(0.25)
q3 = df['price'].quantile(0.75)
iqr = q3 - q1
lim_superior = q3 + 1.5 * iqr
print(f"Limite Superior do Boxplot: ${lim_superior:.2f}")
print(f"Carros considerados outliers estatísticos: {(df['price'] > lim_superior).sum()} unidades.")
print("Decisão analítica: Manter os outliers, pois representam marcas e modelos esportivos de luxo reais (Jaguar, Porsche, BMW), e não erro de digitação.")

#### 3.6 O Que Mais Influencia o Preço? (Correlação com a Variável Alvo)

In [ ]:
# Ranquear variáveis numéricas mais correlacionadas com price
corr_numericas = df.select_dtypes(include="number").corr()
ranking_preco = corr_numericas['price'].sort_values(ascending=False)

print("=== CORRELAÇÃO DE CADA VARIÁVEL COM 'PRICE' ===")
display(ranking_preco)

In [ ]:
# Heatmap das correlações numéricas
plt.figure(figsize=(12, 8))
sns.heatmap(corr_numericas, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title('Matriz de Correlação das Variáveis Numéricas', fontsize=14)
plt.tight_layout()
plt.show()

#### 3.7 Relação do Preço por Categoria de Carroceria

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='carbody', y='price', palette='Set2')
plt.title('Distribuição de Preço por Tipo de Carroceria (carbody)', fontsize=13)
plt.xlabel('Carroceria')
plt.ylabel('Preço ($)')
plt.tight_layout()
plt.show()

#### 3.8 Desafio 3 — Comparação de Tendência entre Variáveis de Alta Correlação
Avaliamos a relação entre `enginesize` (tamanho do motor), `price` e tipo de combustível, além da linha de tendência de `carwidth` (largura do carro).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Dispersão com Hue por combustível
sns.scatterplot(
    data=df,
    x='enginesize',
    y='price',
    hue='fueltype',
    palette={'gas': '#e74c3c', 'diesel': '#27ae60'},
    s=70,
    ax=axes[0]
)
axes[0].set_title('Desafio 3A: Engine Size vs. Price (por Combustível)', fontsize=13)
axes[0].set_xlabel('Tamanho do Motor (enginesize)')
axes[0].set_ylabel('Preço ($)')

# Regplot com reta de regressão linear para carwidth
sns.regplot(
    data=df,
    x='carwidth',
    y='price',
    scatter_kws={'alpha': 0.6, 'color': '#2980b9'},
    line_kws={'color': '#c0392b', 'linewidth': 2},
    ax=axes[1]
)
axes[1].set_title('Desafio 3B: Car Width vs. Price (Linha de Tendência)', fontsize=13)
axes[1].set_xlabel('Largura do Carro (carwidth)')
axes[1].set_ylabel('Preço ($)')

plt.tight_layout()
plt.show()

---
### Fase 4 — Do EDA para o Modelo (Feature Engineering)
Criamos variáveis derivadas a partir dos insights de domínio automotivo obtidos durante a análise.

In [ ]:
# Potência por peso: relação cavalo-vapor / peso do veículo
df['pot_peso'] = df['horsepower'] / df['curbweight']

corr_nova = df[['pot_peso', 'horsepower', 'curbweight', 'price']].corr()['price']
print("Correlação da nova variável 'pot_peso' com o preço:")
print(corr_nova)

plt.figure(figsize=(8, 4))
sns.regplot(data=df, x='pot_peso', y='price', color='#8e44ad')
plt.title('Relação Potência/Peso vs. Preço do Carro', fontsize=13)
plt.xlabel('Potência / Peso (horsepower / curbweight)')
plt.ylabel('Preço ($)')
plt.tight_layout()
plt.show()

### Respostas às Perguntas do Domínio
1. **O que mais encarece um carro?** Variáveis de porte físico e motor (`enginesize`, `curbweight`, `horsepower` e `carwidth`) apresentam altíssima correlação positiva (> 0.75) com o preço.
2. **A marca pesa no preço?** Sim! Marcas premium como Jaguar, Porsche e BMW formam uma faixa de preço nitidamente descolada das montadoras de volume populares (Toyota, Honda, Mazda).
3. **Existe um segmento de luxo separado do resto?** Sim, carros com preços acima de R$ 30.000 formam um cluster evidente com alta motorização e dimensões maiores.
